# Module 2 — Mac Local AI Development Environment

Companion notebook to [`docs/modules/02_mac_local_ai_development_environment.md`](../docs/modules/02_mac_local_ai_development_environment.md).

**Machine note:** the Mac this notebook is being run on deliberately has no local model
runtime installed (limited disk/memory — see `PROGRESS.md`). Every cell below is read-only
with respect to the system: it checks for tools/runtimes and reports what it finds, and
never installs, downloads, or pulls anything. On a properly resourced Mac, re-running this
notebook after `bash scripts/module_02/setup_mac.sh` will produce real smoke-test output
instead of skip notices.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_02"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Machine profile and dev tool check (Lab 2.1)

This is read-only: it looks for `git`, `make`, `cmake`, `uv`, `jq`, `rg` (required) and
`ollama`, `brew` (optional/informational) on `PATH` and reports their versions.

In [2]:
from mac_environment_check import (
    check_all_tools,
    detect_machine_profile,
    missing_required_tools,
    tools_to_markdown_table,
)
from IPython.display import Markdown, display

profile = detect_machine_profile()
print(f"Architecture: {profile.architecture} ({'Apple Silicon' if profile.is_apple_silicon else 'not Apple Silicon'})")
print(f"macOS version: {profile.macos_version}")
print(f"Python version: {profile.python_version}")

checks = check_all_tools()
display(Markdown(tools_to_markdown_table(checks)))

missing = missing_required_tools(checks)
if missing:
    print(f"Missing required tools: {missing}")
else:
    print("All required dev tools are present on this machine.")

Architecture: arm64 (Apple Silicon)
macOS version: 13.0
Python version: 3.12.1


| Tool | Required | Found | Path | Version |
|---|---|---|---|---|
| git | yes | yes | /usr/bin/git | git version 2.39.2 (Apple Git-143) |
| make | yes | yes | /usr/bin/make | GNU Make 3.81 |
| cmake | yes | yes | /opt/homebrew/bin/cmake | cmake version 4.3.4 |
| uv | yes | yes | /Library/Frameworks/Python.framework/Versions/3.12/bin/uv | uv 0.6.8 (c1ef48276 2025-03-18) |
| jq | yes | yes | /opt/homebrew/bin/jq | jq-1.7.1 |
| rg | yes | no | n/a | n/a |
| ollama | no | no | n/a | n/a |
| brew | no | yes | /opt/homebrew/bin/brew | Homebrew 6.0.5 |

Missing required tools: ['rg']


## 2. Model cache report (theory §10-12)

Scans the default cache locations for Ollama and Hugging Face (used by `mlx-lm`).
llama.cpp/llama-cpp-python has no default cache — you point it at wherever you saved a
`.gguf` file directly, so there's nothing to scan for that runtime.

In [3]:
from model_cache import scan_caches, caches_to_markdown_table

locations = scan_caches()
display(Markdown(caches_to_markdown_table(locations)))

| Runtime | Cache path | Exists | Size |
|---|---|---|---:|
| ollama | `/Users/bhakti/.ollama/models` | no | n/a |
| huggingface (used by mlx-lm) | `/Users/bhakti/.cache/huggingface/hub` | yes | 2.5 GB |

## 3. Runtime smoke tests (Labs 2.2-2.4)

Each cell below either produces a real measurement (on a machine with that runtime
installed) or an explicit, actionable skip message (on this machine). None of them install
anything.

In [4]:
# Lab 2.2 - Ollama
import smoke_test_ollama

smoke_test_ollama.run(model=smoke_test_ollama.DEFAULT_MODEL, prompt=smoke_test_ollama.DEFAULT_PROMPT)

SKIPPED: Ollama is not reachable at http://localhost:11434.

To complete this lab on a resourced machine:
  brew install ollama
  brew services start ollama          # or: ollama serve
  ollama pull qwen2.5:1.5b
  uv run python scripts/module_02/smoke_test_ollama.py --model qwen2.5:1.5b



1

In [5]:
# Lab 2.3 - llama-cpp-python OpenAI-compatible server
import smoke_test_llamacpp_server

smoke_test_llamacpp_server.run(
    base_url=smoke_test_llamacpp_server.DEFAULT_BASE_URL,
    prompt=smoke_test_llamacpp_server.DEFAULT_PROMPT,
    model="local-gguf-model",
)

SKIPPED: the `openai` package is not installed. Run: uv add openai


1

In [6]:
# Lab 2.4 - MLX
import smoke_test_mlx

smoke_test_mlx.run(model=smoke_test_mlx.DEFAULT_MODEL, prompt=smoke_test_mlx.DEFAULT_PROMPT)

SKIPPED: `mlx_lm` is not importable.

To complete this lab on a resourced Apple Silicon Mac:
  uv add mlx mlx-lm
  uv run python scripts/module_02/smoke_test_mlx.py

Note: the first run downloads the model from Hugging Face into
~/.cache/huggingface/hub — see model_cache.py for sizing that cache.



1

## 4. Generate the full deliverable report

Running the cell below (or `uv run python scripts/module_02/smoke_test_runtimes.py`) is what
produced [`reports/module_02_environment_report.md`](../reports/module_02_environment_report.md).

In [7]:
import smoke_test_runtimes

report = smoke_test_runtimes.build_report()
display(Markdown(report))

# Module 2 — environment smoke test report

## Machine profile

- Architecture: arm64 (Apple Silicon)
- macOS version: 13.0
- Python version: 3.12.1

## Dev tool check

| Tool | Required | Found | Path | Version |
|---|---|---|---|---|
| git | yes | yes | /usr/bin/git | git version 2.39.2 (Apple Git-143) |
| make | yes | yes | /usr/bin/make | GNU Make 3.81 |
| cmake | yes | yes | /opt/homebrew/bin/cmake | cmake version 4.3.4 |
| uv | yes | yes | /Library/Frameworks/Python.framework/Versions/3.12/bin/uv | uv 0.6.8 (c1ef48276 2025-03-18) |
| jq | yes | yes | /opt/homebrew/bin/jq | jq-1.7.1 |
| rg | yes | no | n/a | n/a |
| ollama | no | no | n/a | n/a |
| brew | no | yes | /opt/homebrew/bin/brew | Homebrew 6.0.5 |

## Model cache report

| Runtime | Cache path | Exists | Size |
|---|---|---|---:|
| ollama | `/Users/bhakti/.ollama/models` | no | n/a |
| huggingface (used by mlx-lm) | `/Users/bhakti/.cache/huggingface/hub` | yes | 2.5 GB |

## Lab 2.2 — Ollama

Exit code: 1

```text
SKIPPED: Ollama is not reachable at http://localhost:11434.

To complete this lab on a resourced machine:
  brew install ollama
  brew services start ollama          # or: ollama serve
  ollama pull qwen2.5:1.5b
  uv run python scripts/module_02/smoke_test_ollama.py --model qwen2.5:1.5b
```

## Lab 2.3 — llama-cpp-python server

Exit code: 1

```text
SKIPPED: the `openai` package is not installed. Run: uv add openai
```

## Lab 2.4 — MLX

Exit code: 1

```text
SKIPPED: `mlx_lm` is not importable.

To complete this lab on a resourced Apple Silicon Mac:
  uv add mlx mlx-lm
  uv run python scripts/module_02/smoke_test_mlx.py

Note: the first run downloads the model from Hugging Face into
~/.cache/huggingface/hub — see model_cache.py for sizing that cache.
```
